In [35]:
import pandas as pd
import os

from pathlib import Path

In [25]:
os.getcwd()

'/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence/notebooks'

In [26]:
os.chdir("/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence")

In [27]:
os.getcwd()

'/Users/gurjitsingh/Desktop/DS_and_AI_projects/Whitefly_Risk_Intelligence'

### Function 1: load_and_validate_trap_metadata(path: Path)

In [13]:
METADATA_REQ_COLS = [
    'site_id', 
    'latitude', 
    'longitude'
]

In [95]:
# Function 1:
def load_and_validate_trap_metadata(path: Path) -> pd.DataFrame:
    """
    Loads and validates the trap metadata
    
    Inputs:
        - Path to trap metadata CSV file

    Outputs:
        - Trap metadata pandas df
    """
    print("Starting Function 1: Loading and validating trap metadata")
    
    # Read the trap metadata and store as data frame
    metadata_df = pd.read_csv(path, dtype={"site_id": "string"})

    # Verify if the parsed dataset is empty
    if metadata_df.empty:
        raise ValueError("The parsed trap metadata df is empty.")

    # Verify all the three required columns are present
    missing_cols = []
    for col in METADATA_REQ_COLS:
        if col not in metadata_df.columns.tolist():
            missing_cols.append(col)

    if len(missing_cols) != 0:
        raise ValueError(f"There are missing required columns in the parsed metadata: {missing_cols}")

    # Strip blank spaces (if any) from each of the site IDs
    metadata_df["site_id"] = metadata_df["site_id"].str.strip()

    # Validate if there are blank spaces as site IDs
    blank_site_ids = metadata_df.loc[metadata_df["site_id"] == "", ]
    if blank_site_ids.shape[0] != 0:
        raise ValueError(f"There are {len(blank_site_ids)} blank site IDs. Row indexes: {blank_site_ids.index.tolist()}")

    # Verify if the site ID column has NULL values
    missing_sites = metadata_df.loc[metadata_df["site_id"].isna(), ]
    if missing_sites.shape[0] != 0:
        raise ValueError(f"The 'site_id' column has {len(missing_sites)} missing values. Row indexes: {missing_sites.index.tolist()}")

    # Verify if there are duplicate site IDs
    duplicated_sites = metadata_df["site_id"].duplicated()
    if duplicated_sites.any():
        dup_values = metadata_df.loc[duplicated_sites, "site_id"].unique()
        raise ValueError(f"There are duplicated 'site_id' column values: {dup_values.tolist()}")

    # Verify if there are missing values in the Latitude column
    missing_latitude = metadata_df["latitude"].isna()
    if missing_latitude.any():
        raise ValueError(f"There are {missing_latitude.sum()} missing values in the 'latitude' column.")

    # Verify if there are missing values in the Longitude column
    missing_longitude = metadata_df["longitude"].isna()
    if missing_longitude.any():
        raise ValueError(f"There are {missing_longitude.sum()} missing values in the 'longitude' column.")

    # Verify if there are non-numeric values in the Latitude column
    # If not, then set the column dtype as float
    bad_latitude = pd.to_numeric(metadata_df["latitude"], errors="coerce").isna()
    if bad_latitude.any():
        bad_lat_rows = metadata_df.loc[bad_latitude, "latitude"]
        raise ValueError(f"There are non-numeric values in the column 'latitude': {bad_lat_rows.tolist()}")

    metadata_df["latitude"] = metadata_df["latitude"].astype("float")

    # Verify if there are non-numeric values in the Longitude column
    # If not, then set the column dtype as float
    bad_longitude = pd.to_numeric(metadata_df["longitude"], errors="coerce").isna()
    if bad_longitude.any():
        bad_lon_rows = metadata_df.loc[bad_longitude, "longitude"]
        raise ValueError(f"There are non-numeric values in the column 'longitude': {bad_lon_rows.tolist()}")

    metadata_df["longitude"] = metadata_df["longitude"].astype("float")

    # Verify if the Latitude values lie in the valid range
    if (metadata_df["latitude"].min() < -90) or (metadata_df["latitude"].max() > 90):
        raise ValueError("Latitude column has values outside the permissible bounds.")

    # Verify if the Longitude values lie in the valid range
    if (metadata_df["longitude"].min() < -180) or (metadata_df["longitude"].max() > 180):
        raise ValueError("Longitude column has values outside the permissible bounds.")

    return metadata_df

In [30]:
df = pd.read_csv("data/interim/trap_metadata.csv")
df.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22
2,29224,Trap 104,31.557827,-83.480183,Active,2020-04-22,2026-06-22
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15


In [31]:
func_1_test_1_df = df.copy()

#### Test 1: Dropping a required column

In [ ]:
func_1_test_1_df = func_1_test_1_df.drop("latitude", axis=1)
func_1_test_1_df.head()

,site_id,trap_label,longitude,status,first_report,last_report
0,29221,Trap 101,-83.465008,Active,2020-04-22,2026-06-22
1,29222,Trap 102,-83.405028,Active,2020-04-22,2026-06-22
2,29224,Trap 104,-83.480183,Active,2020-04-22,2026-06-22
3,29225,Trap 105,-83.573040,Active,2020-04-22,2026-06-15
4,29226,Trap 106,-83.662878,Active,2020-04-22,2026-06-15


In [33]:
func_1_test_1_df.to_csv("data/interim/func_1_test_1_df.csv", index=False)

In [37]:
path_1 = Path("data/interim/func_1_test_1_df.csv")
load_and_validate_trap_metadata(path_1)

Starting Function 1: Loading and validating trap metadata


ValueError: There are missing required columns in the parsed metadata: ['latitude']

#### Test 2: Set one site_id to an empty blank space string

In [69]:
func_1_test_2_df = df.copy()

# First set the dtype of the column as string
func_1_test_2_df["site_id"] = func_1_test_2_df["site_id"].astype("string")

func_1_test_2_df.iloc[1, 0] = " "
func_1_test_2_df.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22
1,,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22
2,29224,Trap 104,31.557827,-83.480183,Active,2020-04-22,2026-06-22
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15


In [70]:
func_1_test_2_df.to_csv("data/interim/func_1_test_2_df.csv", index=False)

In [71]:
path_2 = Path("data/interim/func_1_test_2_df.csv")
load_and_validate_trap_metadata(path_2)

Starting Function 1: Loading and validating trap metadata


ValueError: There are 1 blank site IDs. Row indexes: [1]

#### Test 3: Set one site_id to a missing value

In [82]:
func_1_test_3_df = df.copy()

func_1_test_3_df.iloc[1:4, 0] = pd.NA
func_1_test_3_df.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221.0,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22
1,NaN,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22
2,NaN,Trap 104,31.557827,-83.480183,Active,2020-04-22,2026-06-22
3,NaN,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15
4,29226.0,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15


In [83]:
func_1_test_3_df.to_csv("data/interim/func_1_test_3_df.csv", index=False)

In [84]:
path_3 = Path("data/interim/func_1_test_3_df.csv")
load_and_validate_trap_metadata(path_3)

Starting Function 1: Loading and validating trap metadata


ValueError: The 'site_id' column has 3 missing values. Row indexes: [1, 2, 3]

#### Test 4: Copy one site_id onto another row

In [86]:
func_1_test_4_df = df.copy()

func_1_test_4_df.iloc[1:3, 0] = func_1_test_4_df.iloc[4,0]
func_1_test_4_df.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22
1,29226,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22
2,29226,Trap 104,31.557827,-83.480183,Active,2020-04-22,2026-06-22
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15


In [87]:
func_1_test_4_df.to_csv("data/interim/func_1_test_4_df.csv", index=False)

In [96]:
path_4 = Path("data/interim/func_1_test_4_df.csv")
load_and_validate_trap_metadata(path_4)

Starting Function 1: Loading and validating trap metadata


ValueError: There are duplicated 'site_id' column values: ['29226']

#### Test 5: Set one latitude to 'abc'

In [97]:
func_1_test_5_df = df.copy()

func_1_test_5_df['latitude'] = func_1_test_5_df['latitude'].astype("string")

func_1_test_5_df.iloc[2,2] = "abc"
func_1_test_5_df.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.46603,-83.465008,Active,2020-04-22,2026-06-22
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22
2,29224,Trap 104,abc,-83.480183,Active,2020-04-22,2026-06-22
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15


In [98]:
func_1_test_5_df.to_csv("data/interim/func_1_test_5_df.csv", index=False)

In [99]:
path_5 = Path("data/interim/func_1_test_5_df.csv")
load_and_validate_trap_metadata(path_5)

Starting Function 1: Loading and validating trap metadata


ValueError: There are non-numeric values in the column 'latitude': ['abc']

#### Test 6: Set one longitude to missing

In [100]:
func_1_test_6_df = df.copy()

func_1_test_6_df.iloc[2,3] = pd.NA
func_1_test_6_df.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22
2,29224,Trap 104,31.557827,NaN,Active,2020-04-22,2026-06-22
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15


In [101]:
func_1_test_6_df.to_csv("data/interim/func_1_test_6_df.csv", index=False)

In [102]:
path_6 = Path("data/interim/func_1_test_6_df.csv")
load_and_validate_trap_metadata(path_6)

Starting Function 1: Loading and validating trap metadata


ValueError: There are 1 missing values in the 'longitude' column.

#### Test 7: Set one latitude to 91

In [103]:
func_1_test_7_df = df.copy()

func_1_test_7_df.iloc[2,2] = 91
func_1_test_7_df.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22
2,29224,Trap 104,91.000000,-83.480183,Active,2020-04-22,2026-06-22
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15


In [104]:
func_1_test_7_df.to_csv("data/interim/func_1_test_7_df.csv", index=False)

In [105]:
path_7 = Path("data/interim/func_1_test_7_df.csv")
load_and_validate_trap_metadata(path_7)

Starting Function 1: Loading and validating trap metadata


ValueError: Latitude column has values outside the permissible bounds.

#### Test 8: Set one longitude to -181

In [106]:
func_1_test_8_df = df.copy()

func_1_test_8_df.iloc[2,3] = -181
func_1_test_8_df.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,2020-04-22,2026-06-22
1,29222,Trap 102,31.465087,-83.405028,Active,2020-04-22,2026-06-22
2,29224,Trap 104,31.557827,-181.000000,Active,2020-04-22,2026-06-22
3,29225,Trap 105,31.541213,-83.573040,Active,2020-04-22,2026-06-15
4,29226,Trap 106,31.524275,-83.662878,Active,2020-04-22,2026-06-15


In [107]:
func_1_test_8_df.to_csv("data/interim/func_1_test_8_df.csv", index=False)

In [108]:
path_8 = Path("data/interim/func_1_test_8_df.csv")
load_and_validate_trap_metadata(path_8)

Starting Function 1: Loading and validating trap metadata


ValueError: Longitude column has values outside the permissible bounds.